In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Business Pipeline & Analytics") \
    .getOrCreate()

In [9]:
import os

print(os.listdir())

['.config', 'sample_data']


In [10]:
from google.colab import files
uploaded = files.upload()

Saving sales.csv to sales.csv


In [11]:
from google.colab import files
uploaded = files.upload()

Saving customers.csv to customers.csv


In [12]:
customers_df = spark.read.csv(
    "customers.csv",
    header=True,
    inferSchema=True
)

sales_df = spark.read.csv(
    "sales.csv",
    header=True,
    inferSchema=True
)

Remove rows with null keys

In [13]:
customers_clean = customers_df.dropna(subset=["customer_id"])
sales_clean = sales_df.dropna(subset=["customer_id"])

Remove duplicate rows

In [14]:
customers_clean = customers_clean.dropDuplicates()

In [15]:
sales_clean = sales_clean.dropDuplicates()

Filter invalid values (negative amounts)

In [16]:
sales_clean = sales_clean.filter(col("total_amount") >= 0)

Check schemas

In [17]:
customers_clean.printSchema()
sales_clean.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: integer (nullable = true)

root
 |-- sale_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)



Join both datasets

In [18]:
joined_df = customers_clean.join(
    sales_clean,
    on="customer_id",
    how="inner"
)

In [19]:
joined_df.show(5)

+-----------+----------+---------+--------------------+------------+-------------+----------+-----+--------+-------+----------+----------+--------+------------+
|customer_id|first_name|last_name|               email|phone_number|      address|      city|state|zip_code|sale_id|product_id| sale_date|quantity|total_amount|
+-----------+----------+---------+--------------------+------------+-------------+----------+-----+--------+-------+----------+----------+--------+------------+
|          7|  Isabella|    Davis|isabella.davis@ic...|    555-0007|404 Spruce St|     Boise|   ID|   83701|     10|        10|21-01-2024|       5|      110.95|
|         24|    Daniel|   Walker|daniel.walker@fas...|    555-0024|2121 Cedar St| St. Louis|   MO|   63101|     28|         8|11-02-2024|       5|       99.95|
|         42|    Elijah|    Young|elijah.young@gmai...|    555-0042|3939 Cedar St|   Houston|   TX|   77001|     46|         6|01-03-2024|       1|        22.5|
|          3|    Olivia|    Brown|

Task 1: Daily Sales.

SQL

In [20]:
sales_clean.createOrReplaceTempView("sales")

spark.sql("""
SELECT
    sale_date,
    SUM(total_amount) AS total_sales
FROM sales
GROUP BY sale_date
ORDER BY sale_date
""").show()

+----------+-----------+
| sale_date|total_sales|
+----------+-----------+
|01-02-2024|      19.99|
|01-03-2024|       22.5|
|02-03-2024|       40.0|
|03-03-2024|       55.5|
|04-03-2024|     109.96|
|05-02-2024|      39.98|
|05-03-2024|      12.99|
|06-02-2024|      29.99|
|07-02-2024|      89.97|
|08-02-2024|      49.98|
|09-02-2024|     119.96|
|10-02-2024|       15.5|
|11-02-2024|      99.95|
|12-02-2024|       40.0|
|13-02-2024|      66.75|
|14-02-2024|      79.96|
|15-01-2024|      39.98|
|15-02-2024|       45.0|
|16-01-2024|       25.0|
|16-02-2024|      35.98|
+----------+-----------+
only showing top 20 rows


PySpark

In [21]:
daily_sales = sales_clean.groupBy("sale_date") \
    .agg(sum("total_amount").alias("total_sales")) \
    .orderBy("sale_date")

daily_sales.show()

+----------+-----------+
| sale_date|total_sales|
+----------+-----------+
|01-02-2024|      19.99|
|01-03-2024|       22.5|
|02-03-2024|       40.0|
|03-03-2024|       55.5|
|04-03-2024|     109.96|
|05-02-2024|      39.98|
|05-03-2024|      12.99|
|06-02-2024|      29.99|
|07-02-2024|      89.97|
|08-02-2024|      49.98|
|09-02-2024|     119.96|
|10-02-2024|       15.5|
|11-02-2024|      99.95|
|12-02-2024|       40.0|
|13-02-2024|      66.75|
|14-02-2024|      79.96|
|15-01-2024|      39.98|
|15-02-2024|       45.0|
|16-01-2024|       25.0|
|16-02-2024|      35.98|
+----------+-----------+
only showing top 20 rows


Task 2: City-wise Revenue
SQL

First, register the joined DataFrame:

In [22]:
joined_df.createOrReplaceTempView("joined")
spark.sql("""
SELECT
    city,
    SUM(total_amount) AS total_revenue
FROM joined
GROUP BY city
ORDER BY total_revenue DESC
""").show()

+-----------+------------------+
|       city|     total_revenue|
+-----------+------------------+
|  Riverside|135.45999999999998|
|     Boston|            119.96|
|Centerville|            114.97|
|  Las Vegas|            112.47|
|      Boise|            110.95|
|   New York|            109.96|
|    Seattle|            104.99|
|  St. Louis|             99.95|
|  San Diego|              95.5|
| Washington|              92.0|
|    Detroit|             89.97|
|    Memphis|             89.96|
|San Antonio|             79.96|
|     Albany|             79.96|
|     Denver|              74.0|
|   Columbus|             71.97|
|Springfield|             69.97|
|  Charlotte|             69.97|
|  Nashville|             67.47|
|  Milwaukee|             66.75|
+-----------+------------------+
only showing top 20 rows


PySpark

In [23]:
city_revenue = joined_df.groupBy("city") \
    .agg(sum("total_amount").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

city_revenue.show()

+-----------+------------------+
|       city|     total_revenue|
+-----------+------------------+
|  Riverside|135.45999999999998|
|     Boston|            119.96|
|Centerville|            114.97|
|  Las Vegas|            112.47|
|      Boise|            110.95|
|   New York|            109.96|
|    Seattle|            104.99|
|  St. Louis|             99.95|
|  San Diego|              95.5|
| Washington|              92.0|
|    Detroit|             89.97|
|    Memphis|             89.96|
|San Antonio|             79.96|
|     Albany|             79.96|
|     Denver|              74.0|
|   Columbus|             71.97|
|Springfield|             69.97|
|  Charlotte|             69.97|
|  Nashville|             67.47|
|  Milwaukee|             66.75|
+-----------+------------------+
only showing top 20 rows


Task 3: Top 5 Customers

SQL

In [24]:
spark.sql("""
SELECT
    CONCAT(first_name, ' ', last_name) AS customer_name,
    SUM(total_amount) AS total_spend
FROM joined
GROUP BY first_name, last_name
ORDER BY total_spend DESC
LIMIT 5
""").show()


+--------------+------------------+
| customer_name|       total_spend|
+--------------+------------------+
|  Liam Johnson|135.45999999999998|
|   Henry Moore|            119.96|
|    Emma Jones|            114.97|
|Isabella Davis|            110.95|
|Charlotte Wood|            109.96|
+--------------+------------------+



PySpark

In [25]:
top5_customers = joined_df.groupBy(
    "first_name",
    "last_name"
).agg(
    sum("total_amount").alias("total_spend")
).orderBy(
    col("total_spend").desc()
)

top5_customers.show(5)

+----------+---------+------------------+
|first_name|last_name|       total_spend|
+----------+---------+------------------+
|      Liam|  Johnson|135.45999999999998|
|     Henry|    Moore|            119.96|
|      Emma|    Jones|            114.97|
|  Isabella|    Davis|            110.95|
| Charlotte|     Wood|            109.96|
+----------+---------+------------------+
only showing top 5 rows


Task 4: Repeat Customers (>1 Order)
SQL

In [26]:
sales_clean.createOrReplaceTempView("sales")
spark.sql("""
SELECT
    customer_id,
    COUNT(sale_id) AS order_count
FROM sales
GROUP BY customer_id
HAVING COUNT(sale_id) > 1
ORDER BY order_count DESC
""").show()

+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|          1|          2|
|          4|          2|
|          2|          2|
|         18|          2|
+-----------+-----------+



PySpark

In [27]:
from pyspark.sql.functions import count

repeat_customers = sales_clean.groupBy("customer_id") \
    .agg(count("sale_id").alias("order_count")) \
    .filter(col("order_count") > 1) \
    .orderBy(col("order_count").desc())

repeat_customers.show()

+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|          1|          2|
|          4|          2|
|          2|          2|
|         18|          2|
+-----------+-----------+



Task 5: Customer Segmentation

SQL

In [28]:
spark.sql("""
SELECT
    CONCAT(first_name,' ',last_name) AS customer_name,
    SUM(total_amount) AS total_spend,
    CASE
        WHEN SUM(total_amount) > 10000 THEN 'Gold'
        WHEN SUM(total_amount) BETWEEN 5000 AND 10000 THEN 'Silver'
        ELSE 'Bronze'
    END AS segment
FROM joined
GROUP BY first_name,last_name
ORDER BY total_spend DESC
""").show()

+----------------+------------------+-------+
|   customer_name|       total_spend|segment|
+----------------+------------------+-------+
|    Liam Johnson|135.45999999999998| Bronze|
|     Henry Moore|            119.96| Bronze|
|      Emma Jones|            114.97| Bronze|
|  Isabella Davis|            110.95| Bronze|
|  Charlotte Wood|            109.96| Bronze|
|   Daniel Walker|             99.95| Bronze|
|  Lucas Mitchell|              92.0| Bronze|
|  Harper Jackson|              92.0| Bronze|
|   Elijah Garcia|             89.97| Bronze|
|      Aria Davis|             89.96| Bronze|
|     Sofia Young|             79.96| Bronze|
|   Sophia Garcia|             79.96| Bronze|
|     David Scott|             71.97| Bronze|
|      John Smith|             69.97| Bronze|
|Penelope Roberts|             67.47| Bronze|
|William Anderson|             67.47| Bronze|
|   Noah Williams|             66.75| Bronze|
|   Matthew Allen|             66.75| Bronze|
|   Jackson White|              60

PySpark

In [29]:
segment_df = joined_df.groupBy(
    "customer_id",
    "first_name",
    "last_name",
    "city"
).agg(
    sum("total_amount").alias("total_spend"),
    count("sale_id").alias("order_count")
)

segment_df = segment_df.withColumn(
    "segment",
    when(col("total_spend") > 10000, "Gold")
    .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
    .otherwise("Bronze")
)

segment_df.show()

+-----------+----------+---------+-------------+-----------+-----------+-------+
|customer_id|first_name|last_name|         city|total_spend|order_count|segment|
+-----------+----------+---------+-------------+-----------+-----------+-------+
|         14|     Ethan|   Taylor|  Minneapolis|       15.0|          1| Bronze|
|         39|      Aria|    Davis|      Memphis|      89.96|          1| Bronze|
|         19|   Madison| Thompson|    Charlotte|      29.99|          1| Bronze|
|         18|    Oliver|   Martin| Indianapolis|      59.97|          2| Bronze|
|         16|   Jackson|    White|      Atlanta|       60.0|          1| Bronze|
|         28|       Leo|     King|    Las Vegas|       45.0|          1| Bronze|
|         36|     Mason| Campbell|    Baltimore|      18.75|          1| Bronze|
|          5|      Noah| Williams|     Lakeside|      66.75|          1| Bronze|
|         32|     James|    Green|San Francisco|       27.5|          1| Bronze|
|         37|     Avery|    

Task 6: Final Reporting Table

SQL

In [31]:
segment_df.createOrReplaceTempView("report")
spark.sql("""
SELECT
    CONCAT(first_name,' ',last_name) AS customer_name,
    city,
    total_spend,
    order_count,
    segment
FROM report
ORDER BY total_spend DESC
""").show()

+----------------+------------+------------------+-----------+-------+
|   customer_name|        city|       total_spend|order_count|segment|
+----------------+------------+------------------+-----------+-------+
|    Liam Johnson|   Riverside|135.45999999999998|          2| Bronze|
|     Henry Moore|      Boston|            119.96|          1| Bronze|
|      Emma Jones| Centerville|            114.97|          2| Bronze|
|  Isabella Davis|       Boise|            110.95|          1| Bronze|
|  Charlotte Wood|    New York|            109.96|          1| Bronze|
|   Daniel Walker|   St. Louis|             99.95|          1| Bronze|
|  Lucas Mitchell|  Washington|              92.0|          1| Bronze|
|  Harper Jackson|     Seattle|              92.0|          1| Bronze|
|   Elijah Garcia|     Detroit|             89.97|          1| Bronze|
|      Aria Davis|     Memphis|             89.96|          1| Bronze|
|   Sophia Garcia|      Albany|             79.96|          1| Bronze|
|     

PySpark

In [32]:
final_df = segment_df.select(
    concat_ws(" ", col("first_name"), col("last_name")).alias("customer_name"),
    "city",
    "total_spend",
    "order_count",
    "segment"
).orderBy(col("total_spend").desc())

final_df.show()

+----------------+------------+------------------+-----------+-------+
|   customer_name|        city|       total_spend|order_count|segment|
+----------------+------------+------------------+-----------+-------+
|    Liam Johnson|   Riverside|135.45999999999998|          2| Bronze|
|     Henry Moore|      Boston|            119.96|          1| Bronze|
|      Emma Jones| Centerville|            114.97|          2| Bronze|
|  Isabella Davis|       Boise|            110.95|          1| Bronze|
|  Charlotte Wood|    New York|            109.96|          1| Bronze|
|   Daniel Walker|   St. Louis|             99.95|          1| Bronze|
|  Lucas Mitchell|  Washington|              92.0|          1| Bronze|
|  Harper Jackson|     Seattle|              92.0|          1| Bronze|
|   Elijah Garcia|     Detroit|             89.97|          1| Bronze|
|      Aria Davis|     Memphis|             89.96|          1| Bronze|
|   Sophia Garcia|      Albany|             79.96|          1| Bronze|
|     

Task 7: Save Output

In [33]:
final_df.write.mode("overwrite").csv("output/report")
final_df.write.mode("overwrite").parquet("output/report_parquet")

Reflection Questions
1. Why is cleaning done before joining tables?

Cleaning ensures the data is accurate and consistent before combining datasets. Removing null values, duplicates, and invalid records prevents incorrect joins and improves the quality of the analysis.

2. What would go wrong if null keys are not removed?

Rows with null join keys (such as customer_id) cannot be matched correctly during joins. This results in missing records, incorrect aggregations, and unreliable business insights.

3. How did you decide the join order?

The customers table contains customer information, while the sales table contains transaction details. Both datasets share customer_id, so they were joined using this common key to combine customer details with their sales transactions.

4. Which step was most difficult and why?

Customer segmentation was the most challenging because it required first calculating the total spending for each customer and then applying business rules to classify them into Gold, Silver, and Bronze segments.

5. How is SQL logic similar to PySpark?

SQL and PySpark perform the same operations such as filtering, joining, grouping, and aggregation. SQL uses query syntax, while PySpark uses the DataFrame API. Both produce equivalent results.

6. What challenges will appear with large data?

Large datasets introduce challenges such as:

Increased execution time
Memory limitations
Data skew during joins
Network overhead during shuffling

These can be addressed using partitioning, caching, broadcast joins, and optimized Spark configurations.

7. Can you explain your pipeline in simple steps?
Read customer and sales datasets.
Clean the data by removing null values, duplicates, and invalid records.
Join the datasets using customer_id.
Calculate business metrics such as daily sales, city-wise revenue, top customers, and repeat customers.
Segment customers into Gold, Silver, and Bronze based on total spending.
Create the final reporting table.
Save the final report for future analysis.